# LeetCode 781 — Rabbits in Forest bằng Python

Link đề: https://leetcode.com/problems/rabbits-in-forest/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Hash Table, Math

Skeleton LeetCode:

```python
class Solution:
    def numRabbits(self, answers: List[int]) -> int:
        ...
```

Ghi chú: bài chỉ cần list, dict và số học. Mọi đáp án trong notebook được đối chiếu với một oracle độc lập (liệt kê mọi cách chia thỏ thành các nhóm màu rồi lấy tối thiểu).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong rừng có một số thỏ chưa biết; ta hỏi `n` con thỏ "**có bao nhiêu con thỏ khác cùng màu với bạn?**" và nhận mảng câu trả lời. Cần tìm **số thỏ ít nhất** có thể có trong rừng.
2. Nắm các công cụ Python: `collections.Counter`, `dict.get`, phép **chia lấy phần nguyên `//`**, **chia làm tròn lên** (`(v + g - 1) // g`, `-(-v // g)`, `math.ceil`), phép `%`, `sorted`, và con trỏ trên mảng đã sắp xếp.
3. Đi từ **mô phỏng từng nhóm màu** → **đếm câu trả lời rồi làm tròn lên số nhóm** → **sắp xếp rồi cắt từng đoạn** → **một lượt với "số chỗ còn trống" của từng nhóm** → **đếm số thỏ "ẩn" cần thêm**.
4. Hiểu mấu chốt: một thỏ trả lời `x` thuộc nhóm màu có **đúng `x + 1` con**; các thỏ trả lời **khác nhau** không thể cùng màu; các thỏ trả lời **giống nhau** nên dồn vào cùng nhóm cho tới khi đầy `x + 1` (tham lam).
5. Tránh các lỗi kinh điển: quên `+ 1` (nhóm có `x + 1` con chứ không phải `x`), chia **làm tròn xuống** thay vì lên, gộp các câu trả lời khác nhau vào cùng một nhóm, và chỉ cộng một nhóm cho mỗi giá trị câu trả lời dù có nhiều hơn `x + 1` thỏ trả lời giống nhau.

## 2. Đề bài & ví dụ

Trong rừng có một số lượng thỏ **chưa biết**. Ta hỏi `n` con thỏ câu hỏi: **"Có bao nhiêu con thỏ khác có cùng màu với bạn?"** và thu được câu trả lời trong mảng nguyên `answers`, với `answers[i]` là câu trả lời của con thỏ thứ `i`.

Cho mảng `answers`, hãy trả về **số lượng thỏ ít nhất** có thể có trong rừng.

**Ví dụ 1**

- Input: `answers = [1,1,2]`
- Output: `5`
- Giải thích: hai thỏ trả lời `1` có thể cùng màu, ví dụ đỏ (mỗi con thấy **một** con khác cùng màu). Thỏ trả lời `2` **không thể** màu đỏ (nếu không các câu trả lời mâu thuẫn), giả sử nó màu xanh. Khi đó phải có thêm **2 con xanh khác** trong rừng mà ta **không hỏi**. Số thỏ ít nhất: `3` con đã trả lời cộng `2` con chưa hỏi, bằng `5`.

**Ví dụ 2**

- Input: `answers = [10,10,10]`
- Output: `11`
- Giải thích: ba thỏ cùng màu, mỗi con thấy `10` con khác cùng màu, nên màu đó có `11` con; ta mới hỏi `3` trong số đó.

**Ràng buộc:**

- `1 <= answers.length <= 1000`
- `0 <= answers[i] < 1000`

Chú ý: ta chỉ hỏi **một số** thỏ, không phải tất cả. Vì thế phải đếm cả những thỏ **chưa được hỏi** nhưng chắc chắn tồn tại.

In [1]:
from typing import List
from collections import Counter
from itertools import combinations
import math
import random
import time

examples = [
    ([1, 1, 2], 5),
    ([10, 10, 10], 11),
]
for answers, expected in examples:
    print(answers, "-> kỳ vọng", expected)

[1, 1, 2] -> kỳ vọng 5
[10, 10, 10] -> kỳ vọng 11


## 3. Kiến thức nền

### 3.1. Đọc đề: câu trả lời `x` nghĩa là nhóm có `x + 1` con

Thỏ nói "còn `x` con khác cùng màu với tôi". Vậy **cả màu đó** có `x` con khác cộng với chính nó: **`x + 1` con**. Ví dụ trả lời `0`: không còn con nào khác cùng màu, màu đó chỉ có **1** con (chính nó). Trả lời `2`: màu đó có `3` con.

Điều này cho ta khái niệm **nhóm màu**: một tập thỏ cùng màu, và mọi thỏ **đã được hỏi** trong nhóm đó phải cho **cùng một câu trả lời** `x = (kích thước nhóm) - 1`.

In [2]:
for x in (0, 1, 2, 10):
    print(f"câu trả lời {x:2d} -> cả màu có {x + 1:2d} con thỏ")

câu trả lời  0 -> cả màu có  1 con thỏ
câu trả lời  1 -> cả màu có  2 con thỏ
câu trả lời  2 -> cả màu có  3 con thỏ
câu trả lời 10 -> cả màu có 11 con thỏ


### 3.2. Đếm bằng `collections.Counter`

`Counter(answers)` đếm số lần mỗi giá trị xuất hiện và trả về đối tượng giống `dict`: `{giá_trị: số_lần}`. `.items()` duyệt các cặp `(x, v)`.

In [3]:
c = Counter([1, 1, 2, 0, 1])
print(c)
for x, v in c.items():
    print(f"có {v} con trả lời {x}")
print("đếm giá trị chưa có:", c[7])   # 0, không báo lỗi

Counter({1: 3, 2: 1, 0: 1})
có 3 con trả lời 1
có 1 con trả lời 2
có 1 con trả lời 0
đếm giá trị chưa có: 0


### 3.3. Chia lấy phần nguyên `//`, phần dư `%`, và chia **làm tròn lên**

- `a // b`: chia **làm tròn xuống** (phần nguyên): `7 // 3 = 2`.
- `a % b`: phần dư: `7 % 3 = 1`.

Muốn **làm tròn lên** `ceil(a / b)` với số nguyên dương, có ba cách tương đương:

1. `math.ceil(a / b)` (dùng số thực, an toàn với số nhỏ);
2. `(a + b - 1) // b` (chỉ dùng số nguyên);
3. `-(-a // b)` (đổi dấu hai lần).

Cách số 2 sẽ được dùng chủ yếu. Nó đúng vì: nếu `a` chia hết cho `b`, `a + b - 1` cho thương `a / b`; nếu dư thì `+ b - 1` đẩy lên đúng một bậc.

In [4]:
b = 3
for a in range(1, 10):
    r1 = math.ceil(a / b)
    r2 = (a + b - 1) // b
    r3 = -(-a // b)
    print(f"a={a}: a//b={a // b}, ceil: {r1} {r2} {r3}")
    assert r1 == r2 == r3

a=1: a//b=0, ceil: 1 1 1
a=2: a//b=0, ceil: 1 1 1
a=3: a//b=1, ceil: 1 1 1
a=4: a//b=1, ceil: 2 2 2
a=5: a//b=1, ceil: 2 2 2
a=6: a//b=2, ceil: 2 2 2
a=7: a//b=2, ceil: 3 3 3
a=8: a//b=2, ceil: 3 3 3
a=9: a//b=3, ceil: 3 3 3


### 3.4. "Đóng gói": cần bao nhiêu hộp?

Nếu có `v` vật và mỗi hộp chứa tối đa `g` vật, thì số hộp ít nhất là `ceil(v / g)`. Bài toán này **y hệt**: `v` thỏ trả lời `x`, mỗi nhóm màu chứa tối đa `g = x + 1` thỏ, cần ít nhất `ceil(v / g)` nhóm, và **mỗi nhóm phải có đủ `g` thỏ** (kể cả những con chưa hỏi) nên tốn `g` thỏ.

In [5]:
v, g = 7, 3          # 7 thỏ trả lời 2 (nhóm 3 con)
groups = (v + g - 1) // g
print("số nhóm:", groups, "-> số thỏ:", groups * g, "(trong đó", groups * g - v, "con chưa hỏi)")

số nhóm: 3 -> số thỏ: 9 (trong đó 2 con chưa hỏi)


### 3.5. Sắp xếp để các giá trị giống nhau đứng cạnh nhau

`sorted(answers)` sắp tăng dần. Sau khi sắp, các câu trả lời bằng nhau nằm **liền nhau**, nên ta có thể cắt từng đoạn nhờ chỉ số. Ví dụ `[2,1,1,2,1]` sau sắp xếp: `[1,1,1,2,2]`.

In [6]:
a = sorted([2, 1, 1, 2, 1])
print(a)
i = 0
while i < len(a):
    j = i
    while j < len(a) and a[j] == a[i]:
        j += 1
    print(f"đoạn của giá trị {a[i]}: chỉ số [{i}, {j}) dài {j - i}")
    i = j

[1, 1, 1, 2, 2]
đoạn của giá trị 1: chỉ số [0, 3) dài 3
đoạn của giá trị 2: chỉ số [3, 5) dài 2


### 3.6. `dict.get(key, mặc_định)` và trạng thái "còn chỗ trống"

`d.get(k, 0)` lấy giá trị của khóa `k`, hoặc `0` nếu chưa có. Ta sẽ dùng một dict để nhớ "với mỗi câu trả lời `x`, nhóm đang mở còn bao nhiêu chỗ trống".

In [7]:
slots = {}
print(slots.get(2, 0))      # 0: chưa có nhóm nào đang mở cho câu trả lời 2
slots[2] = 2                # mở nhóm: 3 chỗ, con đầu chiếm 1 -> còn 2
slots[2] -= 1
print(slots)

0
{2: 1}


## 4. Ý tưởng nền tảng / trực giác

**Ví dụ 1: `[1, 1, 2]`.**

- Hai thỏ nói `1`. Mỗi con thấy đúng một con khác cùng màu. Nếu cả hai **cùng màu** thì mỗi con thấy con kia: nhóm `{A, B}` có `2` con, đúng với `1 + 1 = 2`. Không cần thỏ ẩn nào.
- Một thỏ nói `2`: nhóm của nó có `3` con. Nhóm này **không thể** trùng với nhóm trên (một nhóm không thể vừa có `2` vừa có `3` con). Nên cần nhóm mới `3` con, ta mới gặp `1`, còn `2` con chưa hỏi.

Tổng: `2 + 3 = 5`.

**Ví dụ: `[1, 1, 1]`.** Ba thỏ nói `1`. Một nhóm chỉ chứa **tối đa `2`** con (`1 + 1`). Hai con vào nhóm 1; con thứ ba **không thể** vào nhóm đã đầy, nên mở nhóm 2 (có `2` con, một con chưa hỏi). Tổng `2 + 2 = 4`.

**Ví dụ: `[0, 0]`.** Mỗi thỏ nói `0` = "không có thỏ nào khác cùng màu", nên mỗi con **một màu riêng**, mỗi nhóm `1` con. Tổng `2`.

**Quy tắc rút ra:** với mỗi giá trị `x` có `v` thỏ trả lời, cần `ceil(v / (x + 1))` nhóm, mỗi nhóm `x + 1` thỏ. Các giá trị `x` khác nhau không chia sẻ nhóm nên cộng độc lập.

In [8]:
def groups_needed(v, x):
    return (v + x) // (x + 1)      # = ceil(v / (x + 1)); (v + (x+1) - 1) // (x+1)

for x, v in [(1, 2), (1, 3), (0, 2), (2, 1), (10, 3), (2, 7)]:
    g = groups_needed(v, x)
    print(f"{v} thỏ nói {x}: nhóm {x + 1} con -> {g} nhóm -> {g * (x + 1)} thỏ")

2 thỏ nói 1: nhóm 2 con -> 1 nhóm -> 2 thỏ
3 thỏ nói 1: nhóm 2 con -> 2 nhóm -> 4 thỏ
2 thỏ nói 0: nhóm 1 con -> 2 nhóm -> 2 thỏ
1 thỏ nói 2: nhóm 3 con -> 1 nhóm -> 3 thỏ
3 thỏ nói 10: nhóm 11 con -> 1 nhóm -> 11 thỏ
7 thỏ nói 2: nhóm 3 con -> 3 nhóm -> 9 thỏ


## 5. Cách 1: Brute Force (mô phỏng từng nhóm màu)

**Ý tưởng:** không dùng công thức. Ta **mô phỏng** việc xếp từng con thỏ vào nhóm màu:

- giữ một **danh sách các nhóm**; mỗi nhóm lưu `(x, số_thỏ_đã_hỏi_trong_nhóm)` — nhóm này dành cho thỏ trả lời `x`, sức chứa `x + 1`;
- với mỗi thỏ trả lời `x`, **tìm** trong danh sách một nhóm có cùng `x` và còn chỗ (`đã_hỏi < x + 1`); nếu có thì cho con thỏ vào; nếu không thì **mở nhóm mới**;
- cuối cùng tổng số thỏ = tổng sức chứa `(x + 1)` của mọi nhóm.

In [9]:
def brute_force(answers):
    groups = []                              # mỗi phần tử [x, số thỏ đã hỏi trong nhóm]
    for x in answers:
        placed = False
        for g in groups:                     # tìm nhóm cùng x còn chỗ
            if g[0] == x and g[1] < x + 1:
                g[1] += 1
                placed = True
                break
        if not placed:
            groups.append([x, 1])            # mở nhóm mới
    return sum(x + 1 for x, _ in groups)

for answers, expected in examples:
    got = brute_force(answers)
    print(answers, "->", got, "| kỳ vọng", expected)
    assert got == expected
print([1, 1, 1], "->", brute_force([1, 1, 1]))
print([0, 0], "->", brute_force([0, 0]))

[1, 1, 2] -> 5 | kỳ vọng 5
[10, 10, 10] -> 11 | kỳ vọng 11
[1, 1, 1] -> 4
[0, 0] -> 2


**Độ phức tạp:** với mỗi con thỏ ta quét danh sách nhóm, số nhóm tối đa `n`. Time **`O(n * G)`** với `G` là số nhóm (tệ nhất `O(n^2)`), Space `O(G)`.

**Nút thắt:** ta quét lại toàn bộ danh sách nhóm cho mỗi con thỏ, trong khi chỉ cần biết "có nhóm mở nào cho câu trả lời `x` còn chỗ không". Ta có thể **đếm trước** số thỏ cho từng `x` và dùng công thức làm tròn lên.

## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét:

1. **Các câu trả lời khác nhau không thể cùng màu.** Nếu thỏ `A` nói `x` và thỏ `B` nói `y` với `x != y` thì hai con không thể cùng màu (cùng màu thì cùng kích thước nhóm, nên cùng câu trả lời). Vì vậy bài toán **tách** thành các bài độc lập theo từng giá trị `x`.
2. **Với cùng `x`, càng dồn nhiều thỏ vào một nhóm càng tốt** (tham lam). Mỗi nhóm có đúng `x + 1` con bất kể bao nhiêu con trong đó đã được hỏi; để ít nhóm nhất ta lấp đầy mỗi nhóm bằng các thỏ đã hỏi: `v` thỏ cần `ceil(v / (x + 1))` nhóm.

Công thức:

```
đáp án = Σ_x  ceil(v_x / (x + 1)) * (x + 1)
```

với `v_x` là số thỏ trả lời `x`.

**Vì sao tham lam đúng?** Mỗi nhóm có kích thước cố định `x + 1` (đã xác định bởi câu trả lời), nên **tổng số thỏ = số nhóm × (x + 1)**. Số nhóm ít nhất là `ceil(v / (x + 1))` vì mỗi nhóm chứa tối đa `x + 1` thỏ đã hỏi. Và đạt được: lấp đầy từng nhóm. Vậy không thể ít hơn.

In [10]:
def formula(answers):
    total = 0
    for x, v in Counter(answers).items():
        g = x + 1
        total += (v + g - 1) // g * g
    return total

for answers, expected in examples:
    print(answers, "->", formula(answers))
    assert formula(answers) == expected
print("[1, 1, 1] ->", formula([1, 1, 1]), "| [0, 0] ->", formula([0, 0]))

[1, 1, 2] -> 5
[10, 10, 10] -> 11
[1, 1, 1] -> 4 | [0, 0] -> 2


## 7. Các cách tiếp theo

Tổng cộng **5 cách**: Cách 1 mô phỏng các nhóm (đã làm); Cách 2 `Counter` + làm tròn lên (cách chính); Cách 3 sắp xếp rồi cắt đoạn; Cách 4 một lượt với "số chỗ còn trống"; Cách 5 đếm số thỏ ẩn cần thêm.

### 7.1. Cách 2: `Counter` + làm tròn lên

**Giải thích từng đoạn:**

1. `cnt = Counter(answers)`: `cnt[x]` là số thỏ trả lời `x`.
2. Với mỗi `(x, v)`: `g = x + 1` (sức chứa nhóm), số nhóm `(v + g - 1) // g`, số thỏ `nhóm * g`.
3. Cộng dồn vào `ans`.

In [11]:
def counter_ceil(answers):
    cnt = Counter(answers)
    ans = 0
    for x, v in cnt.items():
        g = x + 1
        ans += (v + g - 1) // g * g
    return ans

for answers, expected in examples:
    assert counter_ceil(answers) == expected
print("Cách 2 chạy đúng 2 ví dụ")

Cách 2 chạy đúng 2 ví dụ


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def numRabbits(self, answers: List[int]) -> int:
        cnt = Counter(answers)
        ans = 0
        for x, v in cnt.items():
            g = x + 1
            ans += (v + g - 1) // g * g
        return ans
```

**Độ phức tạp:** đếm `O(n)`, duyệt các giá trị khác nhau `O(n)`. Time **`O(n)`**, Space `O(n)`.

### 7.2. Cách 3: Sắp xếp rồi cắt đoạn

**Ý tưởng khác:** không dùng dict. Sắp xếp `answers` để các giá trị giống nhau đứng cạnh nhau. Duyệt bằng chỉ số `i`: lấy `x = a[i]`, **mở một nhóm** (cộng `x + 1` vào đáp án), rồi **nhảy qua** các thỏ cùng giá trị `x` tối đa `x + 1` con (những con này vào nhóm vừa mở). Gặp thỏ khác hoặc hết `x + 1` con thì mở nhóm kế.

**Giải thích từng đoạn:**

1. `a = sorted(answers)`, `i = 0`.
2. Vòng `while i < n`: `x = a[i]`, `ans += x + 1`.
3. `j = i`; trong khi `j < n` và `a[j] == x` và `j - i < x + 1` thì `j += 1` (nhận thỏ vào nhóm).
4. Đặt `i = j` và lặp.

In [12]:
def sort_scan(answers):
    a = sorted(answers)
    n = len(a)
    ans = 0
    i = 0
    while i < n:
        x = a[i]
        ans += x + 1                              # mở một nhóm: x + 1 con
        j = i
        while j < n and a[j] == x and j - i < x + 1:
            j += 1                                # các thỏ cùng x vào nhóm này
        i = j
    return ans

for answers, expected in examples:
    assert sort_scan(answers) == expected
print("Cách 3 chạy đúng 2 ví dụ")

Cách 3 chạy đúng 2 ví dụ


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def numRabbits(self, answers: List[int]) -> int:
        a = sorted(answers)
        n = len(a)
        ans = i = 0
        while i < n:
            x = a[i]
            ans += x + 1
            j = i
            while j < n and a[j] == x and j - i < x + 1:
                j += 1
            i = j
        return ans
```

**Độ phức tạp:** sắp xếp `O(n log n)`, quét `O(n)` (mỗi `j` tiến một lần). Time `O(n log n)`, Space `O(n)` cho bản sắp xếp (hoặc `O(1)` nếu sắp tại chỗ).

### 7.3. Cách 4: Một lượt với "số chỗ còn trống" (online / luồng)

**Ý tưởng:** không cần đếm trước. Giữ dict `free[x]` = số **chỗ trống** của nhóm đang mở cho câu trả lời `x` (một con thỏ đã hỏi sẽ chiếm một chỗ). Với mỗi con thỏ trả lời `x`:

- nếu `free.get(x, 0) > 0`: con này vào nhóm đang mở, `free[x] -= 1`;
- ngược lại mở nhóm mới: cộng `x + 1` vào đáp án; nhóm có `x + 1` chỗ, con này chiếm một nên `free[x] = x`.

Cách này xử lý được dữ liệu **đến lần lượt** (stream), không cần biết trước toàn bộ.

In [13]:
def one_pass_slots(answers):
    free = {}                     # free[x] = số chỗ còn trống ở nhóm đang mở cho câu trả lời x
    ans = 0
    for x in answers:
        if free.get(x, 0) > 0:
            free[x] -= 1          # vào nhóm đang mở
        else:
            ans += x + 1          # mở nhóm mới có x + 1 con
            free[x] = x           # con này chiếm 1 chỗ, còn x chỗ
    return ans

for answers, expected in examples:
    assert one_pass_slots(answers) == expected
print("Cách 4 chạy đúng 2 ví dụ")

Cách 4 chạy đúng 2 ví dụ


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def numRabbits(self, answers: List[int]) -> int:
        free = {}
        ans = 0
        for x in answers:
            if free.get(x, 0) > 0:
                free[x] -= 1
            else:
                ans += x + 1
                free[x] = x
        return ans
```

**Độ phức tạp:** một lượt duyệt, mỗi bước `O(1)`: Time `O(n)`, Space `O(n)` (số giá trị khác nhau tối đa `n`).

### 7.4. Cách 5: Đếm số thỏ "ẩn" cần thêm

**Ý tưởng:** đáp án = (số thỏ đã hỏi `n`) + (số thỏ **chưa hỏi** nhưng chắc chắn có). Với mỗi `x` có `v` thỏ trả lời, nhóm `g = x + 1`: số "ô trống" còn thiếu để lấp đầy nhóm cuối là `(-v) % g` (nếu `v` chia hết `g` thì `0`; nếu dư `r` thì thiếu `g - r`). Vậy:

```
đáp án = n + Σ_x ( (-v_x) % (x + 1) )
```

Ta dùng **mảng đếm** kích thước `1000` (vì `answers[i] < 1000`) thay vì dict.

**Giải thích từng đoạn:**

1. `count = [0] * 1000`; `count[x] += 1` cho mỗi câu trả lời.
2. `hidden = Σ (-count[x]) % (x + 1)` trên các `x` có `count[x] > 0`.
3. Trả `n + hidden`.

Trong Python `(-v) % g` luôn không âm (khác C++/Java), nên viết gọn được; ở ngôn ngữ khác dùng `(g - v % g) % g`.

In [14]:
def hidden_rabbits(answers):
    count = [0] * 1000
    for x in answers:
        count[x] += 1
    hidden = 0
    for x, v in enumerate(count):
        if v:
            hidden += (-v) % (x + 1)       # số thỏ chưa hỏi cần thêm để đủ nhóm cuối
    return len(answers) + hidden

for answers, expected in examples:
    assert hidden_rabbits(answers) == expected
print("Cách 5 chạy đúng 2 ví dụ")
print("ví dụ 1 có", hidden_rabbits([1, 1, 2]) - 3, "con thỏ chưa hỏi")

Cách 5 chạy đúng 2 ví dụ
ví dụ 1 có 2 con thỏ chưa hỏi


**Code nộp LeetCode (Cách 5):**

```python
class Solution:
    def numRabbits(self, answers: List[int]) -> int:
        count = [0] * 1000
        for x in answers:
            count[x] += 1
        hidden = 0
        for x, v in enumerate(count):
            if v:
                hidden += (-v) % (x + 1)
        return len(answers) + hidden
```

**Độ phức tạp:** Time `O(n + 1000)` tức `O(n + M)` với `M` là giá trị lớn nhất; Space `O(M)`. Ưu điểm: không băm, và cho biết trực tiếp **số thỏ chưa hỏi**.

## 8. TRACE từng bước

### 8.1. Trace Cách 2 với `answers = [2, 1, 1, 2, 1, 0, 2]`

Đếm: `2` xuất hiện 3 lần, `1` xuất hiện 3 lần, `0` xuất hiện 1 lần.

| `x` | `v` | `g = x + 1` | số nhóm `ceil(v/g)` | thỏ (`nhóm * g`) | thỏ chưa hỏi |
|---|---|---|---|---|---|
| 2 | 3 | 3 | 1 | 3 | 0 |
| 1 | 3 | 2 | 2 | 4 | 1 |
| 0 | 1 | 1 | 1 | 1 | 0 |

Tổng `3 + 4 + 1 = 8`.

In [15]:
def trace_counter(answers):
    cnt = Counter(answers)
    print("đếm:", dict(cnt))
    total = 0
    for x, v in sorted(cnt.items()):
        g = x + 1
        groups = (v + g - 1) // g
        rabbits = groups * g
        print(f"x={x}: v={v}, g={g}, nhóm=ceil({v}/{g})={groups}, thỏ={rabbits}, chưa hỏi={rabbits - v}")
        total += rabbits
    print("TỔNG =", total)
    return total

trace_counter([2, 1, 1, 2, 1, 0, 2])

đếm: {2: 3, 1: 3, 0: 1}
x=0: v=1, g=1, nhóm=ceil(1/1)=1, thỏ=1, chưa hỏi=0
x=1: v=3, g=2, nhóm=ceil(3/2)=2, thỏ=4, chưa hỏi=1
x=2: v=3, g=3, nhóm=ceil(3/3)=1, thỏ=3, chưa hỏi=0
TỔNG = 8


8

### 8.2. Trace Cách 4 (một lượt, theo thứ tự đến) với `[1, 1, 1, 0, 1, 2]`

| Thỏ | Câu trả lời | `free` trước | Hành động | `ans` | `free` sau |
|---|---|---|---|---|---|
| 1 | 1 | `{}` | mở nhóm mới, `+2` | 2 | `{1:1}` |
| 2 | 1 | `{1:1}` | vào nhóm, `free[1]-=1` | 2 | `{1:0}` |
| 3 | 1 | `{1:0}` | hết chỗ: mở nhóm mới, `+2` | 4 | `{1:1}` |
| 4 | 0 | `{1:1}` | mở nhóm mới, `+1` | 5 | `{1:1, 0:0}` |
| 5 | 1 | `{1:1,0:0}` | vào nhóm đang mở | 5 | `{1:0, 0:0}` |
| 6 | 2 | | mở nhóm mới, `+3` | 8 | `{..., 2:2}` |

In [16]:
def trace_one_pass(answers):
    free, ans = {}, 0
    for t, x in enumerate(answers, 1):
        before = dict(free)
        if free.get(x, 0) > 0:
            free[x] -= 1
            action = f"vào nhóm đang mở (free[{x}] -> {free[x]})"
        else:
            ans += x + 1
            free[x] = x
            action = f"mở nhóm mới (+{x + 1}), free[{x}] = {x}"
        print(f"thỏ {t}: trả lời {x} | free trước {before} | {action} | ans = {ans}")
    return ans

res = trace_one_pass([1, 1, 1, 0, 1, 2])
assert res == 8 == formula([1, 1, 1, 0, 1, 2])

thỏ 1: trả lời 1 | free trước {} | mở nhóm mới (+2), free[1] = 1 | ans = 2
thỏ 2: trả lời 1 | free trước {1: 1} | vào nhóm đang mở (free[1] -> 0) | ans = 2
thỏ 3: trả lời 1 | free trước {1: 0} | mở nhóm mới (+2), free[1] = 1 | ans = 4
thỏ 4: trả lời 0 | free trước {1: 1} | mở nhóm mới (+1), free[0] = 0 | ans = 5
thỏ 5: trả lời 1 | free trước {1: 1, 0: 0} | vào nhóm đang mở (free[1] -> 0) | ans = 5
thỏ 6: trả lời 2 | free trước {1: 0, 0: 0} | mở nhóm mới (+3), free[2] = 2 | ans = 8


### 8.3. Trace Cách 3 (sắp xếp, cắt đoạn) với `[10, 10, 10]` và `[1, 1, 1, 1, 1]`

`[1,1,1,1,1]`: `x = 1`, nhóm 2 con. Đoạn 1: chỉ số `0..1` (2 thỏ), `+2`; đoạn 2: chỉ số `2..3`, `+2`; đoạn 3: chỉ số `4`, `+2`. Tổng `6`.

In [17]:
def trace_sort_scan(answers):
    a = sorted(answers)
    print("đã sắp:", a)
    n, ans, i = len(a), 0, 0
    while i < n:
        x = a[i]
        ans += x + 1
        j = i
        while j < n and a[j] == x and j - i < x + 1:
            j += 1
        print(f"nhóm mới cho x={x}: nhận chỉ số [{i}, {j}) = {a[i:j]}, +{x + 1} -> ans = {ans}")
        i = j
    return ans

assert trace_sort_scan([1, 1, 1, 1, 1]) == 6
assert trace_sort_scan([10, 10, 10]) == 11

đã sắp: [1, 1, 1, 1, 1]
nhóm mới cho x=1: nhận chỉ số [0, 2) = [1, 1], +2 -> ans = 2
nhóm mới cho x=1: nhận chỉ số [2, 4) = [1, 1], +2 -> ans = 4
nhóm mới cho x=1: nhận chỉ số [4, 5) = [1], +2 -> ans = 6
đã sắp: [10, 10, 10]
nhóm mới cho x=10: nhận chỉ số [0, 3) = [10, 10, 10], +11 -> ans = 11


## 9. Code nộp LeetCode

Ta nộp **Cách 2** (`Counter` + làm tròn lên): ngắn nhất, đúng ý nghĩa nhất.

In [18]:
class Solution:
    def numRabbits(self, answers: List[int]) -> int:
        cnt = Counter(answers)                     # cnt[x] = số thỏ trả lời x
        ans = 0
        for x, v in cnt.items():
            g = x + 1                              # mỗi nhóm màu có g con
            ans += (v + g - 1) // g * g            # ceil(v / g) nhóm, mỗi nhóm g con
        return ans


sol = Solution()
for answers, expected in examples:
    got = sol.numRabbits(answers)
    print(answers, "->", got, "| kỳ vọng", expected)
    assert got == expected

[1, 1, 2] -> 5 | kỳ vọng 5
[10, 10, 10] -> 11 | kỳ vọng 11


## 10. Vì sao đúng

Gọi `v_x` là số thỏ trả lời `x`.

**Bước 1 — tách theo giá trị.** Hai thỏ cùng màu thuộc cùng một nhóm kích thước `s`, và mỗi con trả lời `s - 1`. Vậy các con cùng nhóm **bắt buộc cùng câu trả lời**. Suy ra mỗi nhóm màu "chứa" các thỏ có đúng một giá trị trả lời `x`, nhóm có kích thước đúng `x + 1`. Bài toán tách thành các bài độc lập cho từng `x`.

**Bước 2 — cận dưới.** Với giá trị `x`, mỗi nhóm có `x + 1` con nên chứa tối đa `x + 1` con thỏ đã hỏi. Để chứa `v_x` con cần ít nhất `ceil(v_x / (x + 1))` nhóm, mỗi nhóm đóng góp **đúng** `x + 1` con vào tổng số thỏ (kể cả con chưa hỏi). Vậy tổng thỏ ứng với `x` **không thể ít hơn** `ceil(v_x/(x+1)) * (x+1)`.

**Bước 3 — đạt được.** Chia `v_x` con vào `ceil(v_x/(x+1))` nhóm, mỗi nhóm nhận tối đa `x + 1` con đã hỏi; nhóm cuối có thể thiếu, ta thêm thỏ chưa hỏi (cùng màu) cho đủ `x + 1`. Cấu hình này hợp lệ (mỗi thỏ thấy đúng `x` con khác cùng màu).

**Bước 4 — cộng.** Các giá trị `x` khác nhau độc lập, nên tổng là tổng các cận.

Kiểm chứng bằng code: dùng **oracle liệt kê mọi cách chia thỏ thành nhóm**: chia tập các thỏ đã hỏi thành các khối (phân hoạch tập hợp). Một phân hoạch **hợp lệ** nếu mọi khối chỉ gồm thỏ cùng câu trả lời `x` và có không quá `x + 1` con; chi phí là tổng `x + 1` mỗi khối; đáp án là chi phí **nhỏ nhất**. Oracle không dùng công thức nào ở trên.

In [19]:
def set_partitions(items):
    """Sinh mọi phân hoạch của list `items` thành các khối (list các list)."""
    if not items:
        yield []
        return
    first, rest = items[0], items[1:]
    for part in set_partitions(rest):
        # đặt `first` vào một khối có sẵn
        for i in range(len(part)):
            yield part[:i] + [[first] + part[i]] + part[i + 1:]
        # hoặc tạo khối mới chỉ có `first`
        yield [[first]] + part

def oracle(answers):
    best = None
    for part in set_partitions(list(answers)):
        ok = True
        cost = 0
        for block in part:
            x = block[0]
            if any(y != x for y in block) or len(block) > x + 1:
                ok = False
                break
            cost += x + 1
        if ok and (best is None or cost < best):
            best = cost
    return best

for answers, expected in examples[:1]:
    assert oracle(answers) == expected
print("oracle([1,1,2]) =", oracle([1, 1, 2]))
print("oracle([1,1,1]) =", oracle([1, 1, 1]), "| oracle([0,0]) =", oracle([0, 0]))
assert oracle([1, 1, 1]) == 4 and oracle([0, 0]) == 2
print("(số phân hoạch của 8 phần tử là 4140: oracle dùng cho n <= 8)")

oracle([1,1,2]) = 5
oracle([1,1,1]) = 4 | oracle([0,0]) = 2
(số phân hoạch của 8 phần tử là 4140: oracle dùng cho n <= 8)


## 11. Time/Space Complexity

Gọi `n = len(answers)`, `M` là giá trị lớn nhất (`< 1000`).

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng các nhóm | `O(n * G)`, tệ nhất `O(n^2)` | `O(G)` |
| 2. `Counter` + làm tròn lên | `O(n)` | `O(n)` |
| 3. Sắp xếp + cắt đoạn | `O(n log n)` | `O(n)` |
| 4. Một lượt, số chỗ còn trống | `O(n)` | `O(n)` |
| 5. Mảng đếm + số thỏ ẩn | `O(n + M)` | `O(M)` |

Với `n <= 1000` mọi cách đều chạy tức thì. Cách 4 có ưu điểm xử lý được **luồng dữ liệu**; Cách 5 cho thông tin "bao nhiêu thỏ chưa hỏi".

## 12. Bộ test

Nhóm test: ví dụ đề; edge case (một thỏ, toàn `0`, câu trả lời lớn nhất `999`, đúng đủ nhóm, thừa một con); phản ví dụ cho các cách làm sai; đối chiếu **5 cách + lớp nộp** với **oracle phân hoạch** (cho `n <= 8`) trên input ngẫu nhiên có seed; và test kích thước lớn nhất (`n = 1000`, giá trị `< 1000`) đối chiếu giữa các cách.

In [20]:
methods = {
    "brute_force": brute_force,
    "counter_ceil": counter_ceil,
    "sort_scan": sort_scan,
    "one_pass_slots": one_pass_slots,
    "hidden_rabbits": hidden_rabbits,
    "Solution": Solution().numRabbits,
}

# 12.1 ví dụ của đề
for answers, expected in examples:
    for name, fn in methods.items():
        assert fn(list(answers)) == expected, name
print("2 ví dụ: cả", len(methods), "cách đúng")

# 12.2 edge case
edge = [
    ([0], 1),                         # một con, màu riêng
    ([0, 0, 0], 3),                   # mỗi con một màu
    ([999], 1000),                    # nhóm cực lớn, chỉ hỏi 1 con
    ([1, 1], 2),                      # vừa đủ một nhóm
    ([1, 1, 1], 4),                   # thừa một con -> mở nhóm thứ hai
    ([2, 2, 2], 3),                   # vừa đủ nhóm 3
    ([2, 2, 2, 2], 6),                # thừa một con -> nhóm thứ hai đủ 3
    ([1, 0, 1, 0, 2], 7),             # 1:2 con -> 2 ; 0:2 con -> 2 ; 2:1 con -> 3
    ([3, 3, 3, 3, 3, 3, 3, 3], 8),    # 8 thỏ nói 3: đúng 2 nhóm 4
    ([5] * 7, 12),                    # 7 thỏ nói 5 -> 2 nhóm 6
]
for answers, expected in edge:
    for name, fn in methods.items():
        assert fn(list(answers)) == expected, (name, answers, fn(list(answers)), expected)
for answers, expected in edge:
    if len(answers) <= 8:
        assert oracle(answers) == expected, answers
print("edge case: OK")

2 ví dụ: cả 6 cách đúng
edge case: OK


In [21]:
# 12.3 phản ví dụ: các cách làm SAI phổ biến
def wrong_no_plus_one(answers):          # quên +1: nhóm có x con thay vì x + 1
    cnt = Counter(answers)
    return sum(-(-v // x) * x if x else v for x, v in cnt.items())

def wrong_floor(answers):                # chia làm tròn xuống
    cnt = Counter(answers)
    return sum((v // (x + 1)) * (x + 1) for x, v in cnt.items())

def wrong_merge_different(answers):      # gộp mọi câu trả lời vào một nhóm lớn nhất
    return max(answers) + 1 if answers else 0

def wrong_one_group_per_value(answers):  # mỗi giá trị chỉ một nhóm dù có nhiều hơn x + 1 con
    return sum(x + 1 for x in set(answers))

def wrong_sum(answers):                  # chỉ cộng mọi câu trả lời cộng 1
    return sum(a + 1 for a in answers)

tests = [[1, 1, 1], [1, 1, 2], [0, 0], [2, 2, 2, 2]]
print(f"{'input':16s} {'đúng':>5s} {'quên+1':>7s} {'floor':>6s} {'gộp':>5s} {'1 nhóm/giá trị':>15s} {'tổng':>5s}")
for t in tests:
    print(f"{str(t):16s} {oracle(t):5d} {wrong_no_plus_one(t):7d} {wrong_floor(t):6d} {wrong_merge_different(t):5d} {wrong_one_group_per_value(t):15d} {wrong_sum(t):5d}")
assert wrong_no_plus_one([1, 1, 1]) != 4
assert wrong_floor([1, 1, 1]) != 4
assert wrong_merge_different([1, 1, 2]) != 5
assert wrong_one_group_per_value([1, 1, 1]) != 4
assert wrong_sum([1, 1, 1]) != 4
print("các cách sai bị phát hiện")

input             đúng  quên+1  floor   gộp  1 nhóm/giá trị  tổng
[1, 1, 1]            4       3      2     2               2     6
[1, 1, 2]            5       4      2     3               5     7
[0, 0]               2       2      2     1               1     2
[2, 2, 2, 2]         6       4      3     3               3    12
các cách sai bị phát hiện


In [22]:
# 12.4 đối chiếu mọi cách với oracle phân hoạch trên input nhỏ ngẫu nhiên (seed cố định)
rng = random.Random(781)
cases = 0
for _ in range(600):
    n = rng.randint(1, 8)
    hi = rng.choice([0, 1, 2, 3, 5])
    answers = [rng.randint(0, hi) for _ in range(n)]
    expected = oracle(answers)
    for name, fn in methods.items():
        assert fn(list(answers)) == expected, (name, answers)
    cases += 1
print("đối chiếu", cases, "input ngẫu nhiên n <= 8 với", len(methods), "cách và oracle: OK")

đối chiếu 600 input ngẫu nhiên n <= 8 với 6 cách và oracle: OK


In [23]:
# 12.5 test lớn đúng ràng buộc: n = 1000, giá trị < 1000: các cách phải khớp nhau
rng = random.Random(1000)
big = {
    "toàn 0": [0] * 1000,
    "toàn 999": [999] * 1000,
    "toàn 1": [1] * 1000,
    "một con 999": [999],
    "ngẫu nhiên nhỏ": [rng.randint(0, 5) for _ in range(1000)],
    "ngẫu nhiên rộng": [rng.randint(0, 999) for _ in range(1000)],
    "giá trị ~ sqrt": [rng.randint(0, 30) for _ in range(1000)],
}
for title, answers in big.items():
    t0 = time.perf_counter()
    res = {name: fn(list(answers)) for name, fn in methods.items()}
    dt = time.perf_counter() - t0
    assert len(set(res.values())) == 1, (title, res)
    print(f"{title:16s} -> {res['Solution']:6d}   ({dt * 1000:.1f} ms cho 6 cách)")
assert methods["Solution"]([0] * 1000) == 1000
assert methods["Solution"]([999] * 1000) == 1000
assert methods["Solution"]([1] * 1000) == 1000
print("test lớn: OK")

toàn 0           ->   1000   (27.1 ms cho 6 cách)
toàn 999         ->   1000   (0.4 ms cho 6 cách)
toàn 1           ->   1000   (20.0 ms cho 6 cách)
một con 999      ->   1000   (0.1 ms cho 6 cách)
ngẫu nhiên nhỏ   ->   1006   (8.5 ms cho 6 cách)
ngẫu nhiên rộng  -> 319179   (11.6 ms cho 6 cách)
giá trị ~ sqrt   ->   1267   (2.4 ms cho 6 cách)
test lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `+ 1` (nhóm có `x` con thay vì `x + 1`)

Câu trả lời `x` nghĩa là **còn `x` con khác**, nên nhóm có `x + 1` con (tính cả chính nó).

In [24]:
print("[1,1,1]  SAI :", wrong_no_plus_one([1, 1, 1]), "(đúng 4)")
print("[1,1,1]  ĐÚNG:", Solution().numRabbits([1, 1, 1]))
print("[0,0]    SAI :", wrong_no_plus_one([0, 0]), "(đúng 2)")

[1,1,1]  SAI : 3 (đúng 4)
[1,1,1]  ĐÚNG: 4
[0,0]    SAI : 2 (đúng 2)


**Vì sao sai:** với `x = 1`, nhóm phải có `2` con, nhưng nếu dùng `x = 1` làm sức chứa thì mỗi thỏ một nhóm `1` con, và tổng `3` sai. **Sửa:** sức chứa `g = x + 1`.

### Lỗi 2: Chia làm tròn **xuống**

`v // g` bỏ qua phần dư: các thỏ dư vẫn cần một nhóm nữa.

In [25]:
print("[1,1,1]  SAI :", wrong_floor([1, 1, 1]), "(đúng 4: 3 con nói 1 cần 2 nhóm)")
print("[1,1,1]  ĐÚNG:", Solution().numRabbits([1, 1, 1]))
print("[1,1,2]  SAI :", wrong_floor([1, 1, 2]), "(đúng 5; con nói 2 bị bỏ sót)")

[1,1,1]  SAI : 2 (đúng 4: 3 con nói 1 cần 2 nhóm)
[1,1,1]  ĐÚNG: 4
[1,1,2]  SAI : 2 (đúng 5; con nói 2 bị bỏ sót)


**Vì sao sai:** `3 // 2 = 1` chỉ tính một nhóm, bỏ thỏ thứ ba. **Sửa:** làm tròn lên: `(v + g - 1) // g`.

### Lỗi 3: Gộp các câu trả lời khác nhau vào cùng một nhóm

Thỏ nói `1` và thỏ nói `2` **không thể cùng màu**: màu của chúng có kích thước khác nhau.

In [26]:
print("[1,1,2]  SAI (lấy max + 1):", wrong_merge_different([1, 1, 2]), "(đúng 5)")
print("[1,1,2]  ĐÚNG            :", Solution().numRabbits([1, 1, 2]))

[1,1,2]  SAI (lấy max + 1): 3 (đúng 5)
[1,1,2]  ĐÚNG            : 5


**Vì sao sai:** `max + 1 = 3` cho rằng cả ba thỏ cùng một màu, nhưng hai con nói `1` thì màu đó chỉ có `2` con. **Sửa:** xử lý từng giá trị `x` độc lập.

### Lỗi 4: Chỉ một nhóm cho mỗi giá trị

Nếu có nhiều hơn `x + 1` thỏ cùng nói `x`, phải có **nhiều nhóm**.

In [27]:
print("[2,2,2,2]  SAI :", wrong_one_group_per_value([2, 2, 2, 2]), "(đúng 6)")
print("[2,2,2,2]  ĐÚNG:", Solution().numRabbits([2, 2, 2, 2]))

[2,2,2,2]  SAI : 3 (đúng 6)
[2,2,2,2]  ĐÚNG: 6


**Vì sao sai:** bốn thỏ nói `2` không thể cùng một nhóm `3` con. **Sửa:** số nhóm là `ceil(v / (x + 1))`, không phải `1`.

### Lỗi 5 (mô phỏng một lượt): quên cho nhóm mới tính vào chỗ trống

Ở Cách 4, khi mở nhóm mới ta đặt `free[x] = x` (đã trừ chỗ của con vừa vào). Nếu đặt `free[x] = x + 1`, nhóm có thêm một chỗ ảo.

In [28]:
def wrong_slots(answers):
    free, ans = {}, 0
    for x in answers:
        if free.get(x, 0) > 0:
            free[x] -= 1
        else:
            ans += x + 1
            free[x] = x + 1          # SAI: con đầu tiên cũng chiếm một chỗ
    return ans

print("[1,1,1]  SAI :", wrong_slots([1, 1, 1]), "(đúng 4)")
print("[1,1,1]  ĐÚNG:", one_pass_slots([1, 1, 1]))
assert wrong_slots([1, 1, 1]) != 4

[1,1,1]  SAI : 2 (đúng 4)
[1,1,1]  ĐÚNG: 4


**Vì sao sai:** với `free[1] = 2` sau con đầu, hai con tiếp theo đều vào được nhóm này; nhóm 2 con bị nhét 3 thỏ đã hỏi. **Sửa:** `free[x] = x`.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra:**

- Cho biết **mỗi phần tử "thấy"** một con số về nhóm của nó, hỏi **tối thiểu/tối đa** kích thước tổng: nghĩ tới **nhóm có kích thước cố định** rồi **đóng gói**.
- Các giá trị khác nhau **không tương tác** → đếm bằng `Counter` và xử lý độc lập từng giá trị.
- Công thức "cần bao nhiêu hộp sức chứa `g` cho `v` vật": **`ceil(v / g)`**.

**Mẹo:**

1. Dịch đề sang ngôn ngữ nhóm: câu trả lời `x` ⇒ nhóm có `x + 1` phần tử.
2. Làm tròn lên bằng số nguyên: `(v + g - 1) // g`; nhớ `g = x + 1`.
3. Để **kiểm tra**, viết oracle phân hoạch tập hợp cho `n` nhỏ.
4. Muốn **số thỏ ẩn**: `n_tổng - n_đã_hỏi`.
5. Nếu đề đổi thành "câu trả lời là số thỏ **kể cả bạn**" thì sức chứa là `x`, không phải `x + 1`.
6. Cùng họ với **LeetCode 621 Task Scheduler** (nhóm kích thước `n + 1`) và **LeetCode 1296** (chia thành các nhóm kích thước `k`).

## 15. Biến thể

### 15.1. Câu trả lời từ **mọi** con thỏ: mảng có nhất quán không?

Nếu ta hỏi **tất cả** thỏ trong rừng thì không có thỏ ẩn, và mảng `answers` **nhất quán** (tồn tại cách tô màu phù hợp) khi và chỉ khi với mọi `x`, số thỏ trả lời `x` chia hết cho `x + 1`.

In [29]:
def is_consistent_full(answers):
    return all(v % (x + 1) == 0 for x, v in Counter(answers).items())

print(is_consistent_full([1, 1, 2, 2, 2]))   # True: hai con 1 (1 nhóm), ba con 2 (1 nhóm)
print(is_consistent_full([1, 1, 1]))          # False: ba con nói 1 không chia hết cho 2
# đối chiếu với định nghĩa: nhất quán khi số thỏ tối thiểu đúng bằng số thỏ đã hỏi
rng = random.Random(15)
for _ in range(500):
    answers = [rng.randint(0, 3) for _ in range(rng.randint(1, 8))]
    assert is_consistent_full(answers) == (oracle(answers) == len(answers))
print("khớp oracle trên 500 mảng ngẫu nhiên")

True
False


khớp oracle trên 500 mảng ngẫu nhiên


### 15.2. Số thỏ có thể là **bao nhiêu** (không chỉ tối thiểu)?

Giả sử có `total` con thỏ. `total` khả thi khi và chỉ khi `total >= numRabbits(answers)`: thỏ dư luôn có thể là các màu riêng lẻ (mỗi con một màu, không ai được hỏi). Không có cận trên.

### 15.3. Nếu câu trả lời là "kích thước nhóm kể cả mình"

Khi đó thỏ trả lời `x` thuộc nhóm `x` con; chỉ cần thay `g = x + 1` bằng `g = x` (và không có câu trả lời `0`).

In [30]:
def num_rabbits_including_self(answers):
    cnt = Counter(answers)
    return sum((v + x - 1) // x * x for x, v in cnt.items())

print(num_rabbits_including_self([2, 2, 3]))   # 2 con nói 2 -> 1 nhóm 2 con; 1 con nói 3 -> 3 con: 5 (như ví dụ 1 dịch +1)
assert num_rabbits_including_self([x + 1 for x in [1, 1, 2]]) == 5

5


### 15.4. Các bài cùng họ

- **LeetCode 621 — Task Scheduler:** các nhóm có kích thước `n + 1`.
- **LeetCode 1296 — Divide Array in Sets of K Consecutive Numbers:** chia thành nhóm kích thước `k`, dùng `Counter` và tham lam.
- **LeetCode 659 — Split Array into Consecutive Subsequences:** tham lam với số lượng đếm.
- **LeetCode 135 — Candy**, **455 — Assign Cookies:** tham lam theo đếm/sắp xếp.
- **LeetCode 767 — Reorganize String:** đếm tần suất rồi xếp theo nhóm.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng nhóm | tìm nhóm còn chỗ cho từng con | `O(n * G)` | `O(G)` | để hiểu đề, baseline |
| 2. `Counter` + làm tròn lên | `ceil(v/(x+1)) * (x+1)` | `O(n)` | `O(n)` | **cách nộp chuẩn** |
| 3. Sắp xếp + cắt đoạn | các giá trị bằng nhau liền nhau, nhảy `x + 1` | `O(n log n)` | `O(n)` | không muốn dùng dict |
| 4. Một lượt, chỗ còn trống | nhớ chỗ trống của nhóm đang mở | `O(n)` | `O(n)` | dữ liệu đến dạng luồng |
| 5. Mảng đếm + thỏ ẩn | `n + Σ (-v) mod (x+1)` | `O(n + M)` | `O(M)` | khi cần đếm số thỏ chưa hỏi |

Kết luận: dùng Cách 2 khi thi; Cách 4 khi đề đổi sang dữ liệu luồng; Cách 5 cho biết trực tiếp lượng thỏ ẩn.

## 17. Bài tập vận dụng

Làm từ dễ đến khó. Đáp án ở **mục 19**. Các cell TODO **chưa được gọi**; bạn tự điền rồi tự chạy.

### Bài 1 (đoán output)

Không chạy code, đoán kết quả:

```python
print(Solution().numRabbits([0, 0, 1, 1, 1]))
print(Solution().numRabbits([4]))
```

### Bài 2 (tính tay)

Với `answers = [2, 2, 2, 2, 2, 2, 2, 3, 3]`: lập bảng `x`, `v`, `g`, số nhóm, số thỏ, rồi cộng.

### Bài 3 (điền TODO)

Điền chỗ trống của Cách 2:

In [31]:
def ex3_counter(answers):
    cnt = Counter(answers)
    ans = 0
    for x, v in cnt.items():
        g = ...                          # TODO 1: mỗi nhóm màu có bao nhiêu con?
        groups = ...                     # TODO 2: số nhóm = làm tròn lên của v / g
        ans += ...                       # TODO 3: số thỏ của nhóm này
    return ans

# Khi đã điền xong, bỏ dấu # ở dòng sau để thử:
# assert ex3_counter([1, 1, 2]) == 5

### Bài 4 (tự viết lại)

Viết lại **Cách 4 (một lượt, dict `free`)** từ đầu, không nhìn bài giảng. Nhớ: khi mở nhóm mới thì `ans += x + 1` và `free[x] = x`.

In [32]:
def ex4_one_pass(answers):
    # TODO: tự viết lời giải một lượt với dict "số chỗ còn trống"
    ...

# Khi đã viết xong, bỏ dấu # để thử:
# assert ex4_one_pass([1, 1, 1, 0, 1, 2]) == 8

### Bài 5 (mini challenge, biến thể)

Viết `is_consistent(answers)`: giả sử `answers` là câu trả lời của **mọi** con thỏ trong rừng. Hàm trả `True` nếu tồn tại cách tô màu phù hợp. Rồi kiểm tra bằng cách so với "số thỏ tối thiểu có bằng đúng số thỏ đã hỏi không" trên nhiều mảng ngẫu nhiên.

In [33]:
def ex5_is_consistent(answers):
    # TODO: True nếu mọi con thỏ có thể được hỏi hết và câu trả lời nhất quán
    ...

# Khi đã viết xong, thử bằng test ở mục 19

## 18. Gợi ý 3 mức

**Bài 3**

- Mức 1 (nhẹ): thỏ trả lời `x` thì cả nhóm màu có bao nhiêu con?
- Mức 2 (vừa): số nhóm cần cho `v` thỏ: `ceil(v / g)`; dùng số nguyên `(v + g - 1) // g`.
- Mức 3 (gần đáp án): `g = x + 1`; `groups = (v + g - 1) // g`; `ans += groups * g`.

**Bài 4**

- Mức 1: `free[x]` là số chỗ trống của nhóm đang mở cho câu trả lời `x`.
- Mức 2: nếu `free.get(x, 0) > 0` thì vào nhóm, `free[x] -= 1`.
- Mức 3: ngược lại `ans += x + 1; free[x] = x`.

**Bài 5**

- Mức 1: nếu không có thỏ ẩn thì mỗi nhóm đều **đầy** bằng các thỏ đã hỏi.
- Mức 2: với mỗi `x`, số thỏ trả lời `x` phải là bội của `x + 1`.
- Mức 3: `all(v % (x + 1) == 0 for x, v in Counter(answers).items())`.

## 19. Đáp án tham khảo

**Bài 1:** `[0,0,1,1,1]`: hai thỏ nói `0` → mỗi con một màu, `2` thỏ. Ba thỏ nói `1` → nhóm 2 con, cần `ceil(3/2) = 2` nhóm → `4` thỏ. Tổng `6`. `[4]` → nhóm `5` con → `5`.

**Bài 2:** `x = 2`: `v = 7`, `g = 3`, nhóm `ceil(7/3) = 3`, thỏ `9` (có `2` con chưa hỏi). `x = 3`: `v = 2`, `g = 4`, nhóm `ceil(2/4) = 1`, thỏ `4` (có `2` con chưa hỏi). Tổng `9 + 4 = 13`.

In [34]:
# Bài 1
assert Solution().numRabbits([0, 0, 1, 1, 1]) == 6
assert Solution().numRabbits([4]) == 5
print("Bài 1:", Solution().numRabbits([0, 0, 1, 1, 1]), Solution().numRabbits([4]))

# Bài 2
a2 = [2, 2, 2, 2, 2, 2, 2, 3, 3]
assert Solution().numRabbits(a2) == 13
print("Bài 2:", Solution().numRabbits(a2))

# Bài 3
def ex3_answer(answers):
    cnt = Counter(answers)
    ans = 0
    for x, v in cnt.items():
        g = x + 1
        groups = (v + g - 1) // g
        ans += groups * g
    return ans

# Bài 4
def ex4_answer(answers):
    free, ans = {}, 0
    for x in answers:
        if free.get(x, 0) > 0:
            free[x] -= 1
        else:
            ans += x + 1
            free[x] = x
    return ans

# Bài 5
def ex5_answer(answers):
    return all(v % (x + 1) == 0 for x, v in Counter(answers).items())

assert ex3_answer([1, 1, 2]) == 5 and ex4_answer([1, 1, 1, 0, 1, 2]) == 8
rng = random.Random(19)
for _ in range(500):
    answers = [rng.randint(0, 4) for _ in range(rng.randint(1, 8))]
    exp = oracle(answers)
    assert ex3_answer(answers) == ex4_answer(answers) == exp
    assert ex5_answer(answers) == (exp == len(answers))
print("Bài 3, 4, 5: khớp oracle phân hoạch trên 500 input ngẫu nhiên")

Bài 1: 6 5
Bài 2: 13


Bài 3, 4, 5: khớp oracle phân hoạch trên 500 input ngẫu nhiên


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "đóng gói nhóm kích thước cố định":**

```python
class Solution:
    def numRabbits(self, answers: List[int]) -> int:
        cnt = Counter(answers)
        ans = 0
        for x, v in cnt.items():
            g = x + 1                       # kích thước một nhóm màu
            ans += (v + g - 1) // g * g     # ceil(v / g) nhóm, mỗi nhóm g con
        return ans
```

**Checklist trước khi submit:**

- [ ] Nhóm màu có **`x + 1`** con (không phải `x`)?
- [ ] Số nhóm làm tròn **lên** `(v + g - 1) // g`?
- [ ] Mỗi giá trị `x` được xử lý **độc lập** (không gộp các câu trả lời khác nhau)?
- [ ] Đã tính cả những con thỏ **chưa hỏi** (mỗi nhóm đủ `x + 1` con)?
- [ ] Đã thử `[0]`, `[0,0,0]`, `[999]`, `[1,1,1]` (thừa một con), `[2,2,2]` (vừa đủ)?
- [ ] Time `O(n)`, Space `O(n)`?

**Ghi nhớ một câu:** *"Trả lời x nghĩa là nhóm có x + 1 con; v con nói x cần ceil(v / (x + 1)) nhóm, mỗi nhóm x + 1 con."*